<a href="https://colab.research.google.com/github/gtzan/synthesizers_cs_perspective/blob/main/Explaining_the_DFT_a_music_perspective_part5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A music and sound exposition of the Discrete Fourier Transform - Part 5

The Discrete Fourier Transform (DFT) is fundamental for any type of music and sound processing using computers. The goal of this notebook is to develop high-level intuition behind the concepts that underlie the DFT and understand how this transformation is used by various algorithms for audio analysis and synthesis. The intended audience is anyone with some knowledge of high school math that is interested in the topic. This is not a detailed mathematical digital signal processing (DSP) exposition. At the end of the notebook a variety of resources for better understanding the DFT and audio processing are provided for further digging.

There are many great resources for understanding the DFT including textbooks, tutorial articles, and videos some of which are listed at the end of this exposition. This material reflects my own personal way of teaching the DFT that has evolved over twenty years of experience with striving to explain the underlying concepts to students from diverse academic backgrounds and disciplines with a focus on how it connects to music information retrieval.

In **Part 1** we looked at how we can create complex sounds by combining sinudoids of different frequencies and amplitudes that change over time (additive synthesis) such as the sound of a bell or even the infamous THX sound. We also can create plausible and recognizable music melodies and tunes that are similar to the chip tunes made by the early game machines and computer sound cards.

In **Part 2** we started looked at the inverse problem of analysis. Given an array of audio samples that correspond to a complex sound or a simple melody we would like to find what are the frequencies and amplitudes of each sinusoidal component. We started by the simple problem of amplitude estimation of a single sinusoid and looked at peak and RMS amplitude. These methods were not very accurate in the presence of interference so we looked a estimating the amplitude through a dot-product (correlation) with **probe/basis** sinusoid with unit amplitude and frequency equal to the frequency we were looking for. This approach was shown to be very robust to the presence of interference but does require knowledge of the frequency you are training to estimate the amplitude for as well as phase alignment between the probe/basis and the target sinusoid.


In **Part 3** we looked at how we can deal with time and phase. For time we can perform **short-time processing** in which we process the audio samples in small chunks that are long enough to contain a few cycles of the sinusoids we are intrested in but short enough to allow us to estimate amplitudes that change over time.

To estimate the amplitude accurately when the **probe/basis** signal and the target sinusoid are not aligned in time we can calculate multiple amplitude estimate for different time shifts and select the maximum amplitude estimate as our amplitude estimate. This approach is called **cross-correlation** and is a fundamental digital signal processing operation with direct equivalents in mamallian auditory systems. It is a computationally costly operation as multiple dot-products need to be computed and the accuracy of the estimation depends on how many time shits are considered. It can also work for other waveforms than sinusoids. However, if we are dealing with sinusoids there is a much more efficient, clever way to estimate the amplitude.

If we are looking for the amplitude of a sinusoidal signal that is part of a complex signal and we know it's frequency we can take use inner-products of array corresponding to the complex signal that contains our "burried" sinusoid with two basis arrays (or probes) one containing a sampled sine signal of the known frequency and one containing a sampled cosine signal of the known frequency. The resulting two ampltitude estimates can be combined to return an estimate of the mangitude of the sinusoid as well as an estimate of the phase. Each of these sinusoids can be considered a "probe" or "basis" and is a building block that can take as input a complex input signal consisting of multiple sinusoidal signals and potentially other interferring sounds and estimate reliably the amplitude and the phase of of a sinusoid with a particular frequency of interest.

In this **Part 4** we looked at how this basis/probe pair of a unit amplitude sine array of samples and a unit amplitude cosine array of samples form an amplitude and phase estimator for a particular frequency. By creating a linearly-spaced grid of frequencies we can create a transform that can change the representation of our signal from the time-domain to the frequency domain. The frequency domain (or spectrum) specifies how we can re-synthesize/re-construct our original time-domain signal by appropriately combining sinusoids with the estimated dot-product amplitudes with the two basis/probe signals from each frequency. These can be converted to actual amplitude and phase. To re-synthesize the sinusoidal components from the frequency domain we do not need to recompute samples of the sine and cosine function but can simply weigh each probe/basis appropriately and add them together. We called this the Discrete Probe Transform and showed that it can perform **perfect reconstruction** i.e we can get back our original input signal from the spectrum (frequency domain) without any loss of information. The Discrete Probe transform is essentially the Discrete Fourier Transform (DFT). In this part we formalize the DFT and start introducing some cool mathematical notation. The pair of basis/probes amplitude/phase estimator for a specific frequency is the basic building block and the key to understanding the DFT.  



# The Discrete Fourier Transform  


We are finally ready to explore the Discrete Fourier Transform which is a fundamental algorithm in Digital Signal Processing.

















First let's look at the typical mathematical notation:

\begin{equation}
X_k = <x, e^{jtk2\pi /N}> = \sum_{t=0}^{N-1} x_t e^{-j t k 2 \pi/N}  
\end{equation}

We start with a finite, length-N segment of a digital signal $x_0, x_1, \dots, x_{N-1}$. We define the inner product as:
\begin{equation}
<x,y> = \sum_{t=0}^{N-1} x_t y_t^*
\end{equation}
and the corresponding basis is N frequency cosine and sine probe pairs
 equally spaced in the desired rage from zero to the sampling frequency:
\begin{equation}
{e^{jtk2\pi/N}} = {cos(tk2\pi/N) + j sin(tk2\pi/N})\;\;\; \text{for} \;\; 0 \leq k \leq N-1
\end{equation}

This is a typical description you can find in a Digital Signal Processing textbook or Wikipedia and contains a bunch of math notation we have not covered but as we will see essentially we are using N "basis/probes" to estimate the corresponding amplitudes and phases of the sinusoids present in a sound mixture based on all the ideas we have covered so far in Part 1, 2, and 3.



In [ ]:
N = 512
t = np.arange(0,N)

# basis cos and sin probe/basis functions for k=1
k = 1
cos_basis = np.cos(k*t*2*np.pi/N)
sin_basis = np.sin(k*t*2*np.pi/N)
plot_basis(t, cos_basis, sin_basis, k)

# basis cos and sin probe/basis functions for k=4
k = 4
cos_basis = np.cos(k*t*2*np.pi/N)
sin_basis = np.sin(k*t*2*np.pi/N)
plot_basis(t, cos_basis, sin_basis, k)


# basis function real and imaginary parts for k=10
k = 10
cos_basis = np.cos(k*t*2*np.pi/N)
sin_basis = np.sin(k*t*2*np.pi/N)

plot_basis(t, cos_basis, sin_basis, k)


\begin{equation}
X_k = <x, e^{jtk2\pi /N}> = \sum_{t=0}^{N-1} x_t e^{-j t k 2 \pi/N}  = \sum_{t=0}^{N-1} x_t cos(tk2\pi/N) + \sum_{t=0}^{N-1} x_t  j sin(tk2\pi/N)
\end{equation}

Looking at the equation above you can see that for a specific k the complex spectrum value X_k can be computed by taking two inner products for the input signal $x_t$ with the real and imaginary parts of the corresponding basis function.

Let's consider as input a cosine signal of the same frequency as a particular pair of basis functions let's say with $k=4$ and look at the signal resulting from the point-wise multiplication of the input with the real part and the imagine part. As you can see the point-wise multiplication with the real part results in a signal that is centered above zero and therefore the inner product is positive (the value is shown with the straight black line in the plot). The point-wise multiplication with the imaginary part results in a signal that is centered at zero and therefore the inner product is zero (the value is shown with the straight black line in the plot). Recall from before when we examined using the inner product with a sinusoid of known frequency to estimate amplitude that the estimated amplitude is simply twice the inner product.

In [ ]:
def plot_product(t,x1,x2):
    prod = np.multiply(x1,x2)
    inner_prod = np.sum(prod) / len(prod)
    plt.figure(figsize=(20,5))
    plt.plot(t, x1,'.', lw=1, color='blue')
    plt.plot(t, x2,'.', lw=1, color='cyan')
    plt.plot(t, np.multiply(x1,x2), lw=4, color='red')
    ip_line = np.empty(len(prod))
    ip_line.fill(inner_prod)
    plt.plot(t, ip_line, lw=4, color='black')
    plt.plot(t, ip_line*2, lw=2, color='black')
    plt.legend(('x', 'x_basis', 'point-wise product', 'inner product', 'amplitude estimate'), loc='upper right')


k=4
a = 0.6
x = a * np.cos(k*t*2*np.pi/N)
x_re = np.cos(k*t*2*np.pi/N)
x_im = np.sin(k*t*2*np.pi/N)
plot_product(t,x,x_re)
plot_product(t,x,x_im)

When the input is a sine wave the inner product with the real part of the basis is zero and the inner product with the imaginary part of the basis is positive.

In [ ]:

x = a * np.sin(k*t*2*np.pi/N)
plot_product(t,x,x_re)
plot_product(t,x,x_im)


Now let's consider the case of using as input a sinusoidal signal of different frequency let's say corresponding to k=10. As you can observe in this case the inner products with both the real and imaginary part of the basis functions are zero.

In [ ]:
k=10
x = a * np.sin(k*t*2*np.pi/N)
plot_product(t,x,x_re)
plot_product(t,x,x_im)

Now consider an input that is a mixure of two sinsoids. Notice that we are still able to correctly estimate the amplitude of the mixture component that corresponds to the basis function we are using.

In [ ]:
k=4
a1 = 0.8
x1 = a1 * np.sin(k*t*2*np.pi/N)
a2 = 0.4
k = 10
x2 = a2 * np.sin(k*t*2*np.pi/N)
x = x1+x2
plot_product(t,x,x_re)
plot_product(t,x,x_im)

Finally lets look at computing the Discrete Fourier Transform directly from the equation and viewing the magnitude spectrum for the mixture signal we examined above. The code is written so that the connection to measuring amplitude and phases using inner products with the real and imaginary parts of each basis element is emphasized and the code does not utilize the complex number type. Notice the normalization by 2/N so that the mangitude spectrum shows the estimated amplitudes of the input signal.

In [ ]:
def pedagogical_dft(x, N):
    X_re = np.zeros(N)       # array holding the real parts of the spectrum
    X_im = np.zeros(N)       # array holding the imaginary values of the spectrum
    for k in np.arange(0,N):
        for t in np.arange(0,N):
            X_re[k] += x[t] * np.cos(t * k * 2 * np.pi / N)   # inner product with real basis k
            X_im[k] += x[t] * np.sin(t * k * 2 * np.pi / N)   # inner product with imaginary basis k
    return (X_re, X_im)

def plot_mag_spectrum(Xmag):
    plt.figure(figsize=(20,5))
    n = np.arange(0,len(Xmag))
    plt.plot(n,Xmag)

N = 512
n = np.arange(0,N)

# Single sinusoid
k=100
x1 = 1.5 * np.sin(k*n*2*np.pi/N)

(X_re, X_im) = pedagogical_dft(x1, N)
Xmag = 2 * np.sqrt(X_re * X_re + X_im * X_im) /N
plot_mag_spectrum(Xmag)

In [ ]:
# Mixture sinusoid input
x2 = np.sin(50*n*2*np.pi/N) + 0.5 * np.sin(100 * n * 2 * np.pi/N) + 0.3 * np.sin(300 * n * 2 * np.pi/N)

(X_re, X_im) = pedagogical_dft(x2, N)
Xmag = 2 * np.sqrt(X_re * X_re + X_im * X_im) / N
plot_mag_spectrum(Xmag)


In [ ]:
X = np.fft.fft(x1)
Xmag = 2 * np.abs(X) / N
plot_mag_spectrum(Xmag)

X = np.fft.fft(x2)
Xmag = 2 * np.abs(X) / N
plot_mag_spectrum(Xmag)

In [ ]:
%%time
import numpy as np
import timeit

def pedagogical_dft(x, N):
    X_re = np.zeros(N)       # array holding the real parts of the spectrum
    X_im = np.zeros(N)       # array holding the imaginary values of the spectrum
    for k in np.arange(0,N):

        for t in np.arange(0,N):
            X_re[k] += x[t] * np.cos(t * k * 2 * np.pi / N)   # inner product with real basis k
            X_im[k] += x[t] * np.sin(t * k * 2 * np.pi / N)   # inner product with imaginary basis k
    return (X_re, X_im)

def test():
    x = np.zeros(512)
    for i in range(10):
      pedagogical_dft(x,512)

test()

In [ ]:
%%time
import numpy as np
import timeit

def test():
    for i in range(10):
      x = np.zeros(512)
      np.fft.fft(x)

test()

In [ ]:
filename = librosa.ex('fishin')
fishin, sr = librosa.load(filename)
ipd.Audio(fishin, rate=sr)

In [ ]:
import scipy.fft as fft
import random

y = trumpet[0:100000]
#y = fishin[0:10000]
complex_spectrum = fft.rfft(y)

magnitude_spectrum = np.abs(complex_spectrum)
print(len(complex_spectrum))
bin = 10

#mags = magnitude_spectrum[bin:bin+10000]
#magnitude_spectrum = np.zeros(len(magnitude_spectrum))
#magnitude_spectrum[bin:bin+10000] = mags

#magnitude_spectrum = np.roll(np.abs(complex_spectrum), -1000)
magnitude_spectrum = np.ones(len(magnitude_spectrum))
# magnitude_spectrum = np.abs(complex_spectrum)
phase_spectrum = np.angle(complex_spectrum)
# phase_spectrum[0:int(len(magnitude_spectrum))] = np.random.uniform(0, 2*np.pi, int(len(magnitude_spectrum)))

reconstructed_complex_spectrum  = magnitude_spectrum*(np.cos(phase_spectrum)+1j*np.sin(phase_spectrum))
reconstructed = fft.irfft(reconstructed_complex_spectrum)
fig = plt.figure(figsize=(6,2))
plt.xlabel("Time in samples")
plt.ylabel("Amplitude")
plt.plot(reconstructed[0:1000], lw=2, color='blue')
plt.plot(y[0:1000], lw=2, color='blue')

plt.show()
ipd.Audio( reconstructed,rate=sr, normalize=True)



In [ ]:
a = np.concatenate([reconstructed, reconstructed, reconstructed], axis=0)
print(a.shape)

In [ ]:
ipd.Audio(a,rate=sr, normalize=True)

In [ ]:
import scipy.signal as signal
audio_signal = trumpet
hopSize = 256
winSize = 1024
offsets = np.arange(0, len(audio_signal), hopSize)

output = np.zeros(len(audio_signal))
spectrogram = []
times = []
for (m,o) in enumerate(offsets):
    frame = audio_signal[o:o+winSize]
    if len(frame) < winSize:
            break  # Ignore incomplete segment
    window = signal.windows.hann(len(frame))
    mag_spectrum = np.abs(np.fft.fft(frame)[:winSize // 2])
    spectrogram.append(mag_spectrum)
    frame = window * frame
    output[o:o+winSize] += frame
    times.append((o + winSize) / sr)

spectrogram = np.array(spectrogram).T
frequencies = np.fft.fftfreq(winSize, d=1/sr)[:winSize // 2]

ipd.Audio(output, rate=sr)

In [ ]:
a = [1, 2, 4,5, 6]
print(a[:3])
winSize = 1024
print(winSize /2)
print(winSize // 2)

In [ ]:
# Plot the spectrogram
plt.figure(figsize=(10, 6))
plt.pcolormesh(times, frequencies, 10 * np.log10(spectrogram + 1e-10), shading='gouraud', cmap='viridis')
#plt.pcolormesh(times, frequencies, spectrogram, shading='gouraud', cmap='viridis')

plt.colorbar(label='Power/Frequency (dB/Hz)')
plt.title('Spectrogram')
plt.ylabel('Frequency (Hz)')
plt.xlabel('Time (s)')
plt.tight_layout()
plt.show()

In [ ]:
# Plot the spectrogram
plt.figure(figsize=(10, 6))
plt.pcolormesh(times, frequencies, spectrogram, shading='gouraud', cmap='viridis')
#plt.pcolormesh(times, frequencies, spectrogram, shading='gouraud', cmap='viridis')

plt.colorbar(label='Power/Frequency (dB/Hz)')
plt.title('Spectrogram')
plt.ylabel('Frequency (Hz)')
plt.xlabel('Time (s)')
plt.tight_layout()
plt.show()

In [ ]:
import scipy.signal as signal
audio_signal = trumpet
hopSize = 1 * 512
winSize = 1 * 4096
offsets = np.arange(0, len(audio_signal), hopSize)

output = np.zeros(len(audio_signal))
spectrogram = []
times = []
for (m,o) in enumerate(offsets):
    frame = audio_signal[o:o+winSize]
    if len(frame) < winSize:
            break  # Ignore incomplete segment
    window = signal.windows.hann(len(frame))
    windows = np.ones(len(frame))
    #mag_spectrum = np.abs(np.fft.fft(frame)[:winSize // 2])
    #spectrogram.append(mag_spectrum
    complex_spectrum = np.fft.fft(window*frame)
    mag_spectrum = np.abs(complex_spectrum)
    sorted_mag_spectrum = np.sort(mag_spectrum)
    threshold = sorted_mag_spectrum[-4]
    mag_spectrum[mag_spectrum < threshold] = 0
    #spectrogram.append(mag_spectrum)
    #mag_spectrum = np.ones(len(frame))
    phase_spectrum = np.angle(complex_spectrum)
    phase_spectrum[0:int(len(mag_spectrum))] = np.random.uniform(0, 2*np.pi, int(len(mag_spectrum)))
    #phase_spectrum = np.zeros(len(mag_spectrum))

    reconstructed_complex_spectrum  = mag_spectrum*(np.cos(phase_spectrum)+1j*np.sin(phase_spectrum))
    reconstructed_frame = np.real(fft.ifft(reconstructed_complex_spectrum))

    reconstructed = window * reconstructed_frame
    output[o:o+winSize] += reconstructed_frame
    times.append((o + winSize) / sr)

spectrogram = np.array(spectrogram).T
frequencies = np.fft.fftfreq(winSize, d=1/sr)[:winSize // 2]
ipd.Audio(output, rate=sr)

In [ ]:
import soundfile as sf
def plot_mag_spectrum(Xmag):
    plt.figure(figsize=(20,5))
    n = np.arange(0,len(Xmag))
    plt.plot(n,Xmag)

# load a C4 sine wave
x,srate = sf.read('sineC4.wav')
print(np.max(x))
ipd.Audio(x,rate=srate)

In [ ]:
N = len(x)
X = np.fft.fft(x)
Xmag = 2 * np.abs(X) / N
plot_mag_spectrum(Xmag)

In [ ]:
# load a C4 sine wave - notice that there are multiple peaks
# corresponding to the harmonics
x,srate = sf.read('cl_C4.wav')
print(np.max(x))
ipd.Audio(x,rate=srate)
ipd.Audio(x,rate=srate)

In [ ]:
N = len(x)
X = np.fft.fft(x)
Xmag = 2 * np.abs(X) / N
plot_mag_spectrum(Xmag)

In [ ]:
# let's zoom to the first 5000 values to see the harmonics better
plot_mag_spectrum(Xmag[1:5000])

In [ ]:
# load a 3 second excerpt from a jazz recording
x, srate = sf.read('freddie3sec.wav')
ipd.Audio(x,rate=srate)

In [ ]:
N = len(x)
X = np.fft.fft(x)
Xmag = 2 * np.abs(X) / N
plot_mag_spectrum(Xmag)
plot_mag_spectrum(Xmag[1:5000])

In [ ]:
def plot_mag_spectrum(Xmag, N):
    plt.figure(figsize=(20,5))
    n = np.arange(0,len(Xmag))
    plt.xlabel('Frequency as DFT bin')
    plt.plot(n,Xmag)

    plt.figure(figsize=(20,5))
    n = np.linspace(0.0, len(Xmag) *1.0 /N, num=len(Xmag))
    plt.xlabel('Frequency in fractions of sampling rate')
    plt.plot(n,Xmag)

    plt.figure(figsize=(20,5))
    n = np.linspace(0.0, (len(Xmag) *1.0 / N)*44100, num=len(Xmag))
    plt.xlabel('Frequency in Hz ')

    plt.plot(n,Xmag)

# load a C4 sine wave
x, srate = sf.read('freddie3sec.wav')
N = len(x)
X = np.fft.fft(x)
Xmag = 2 * np.abs(X) / N
plot_mag_spectrum(Xmag[1:5000], len(Xmag) )

In [ ]:
# load a C4 sine wave
x, srate = sf.read('sineC4.wav')
N = len(x)
X = np.fft.fft(x)
Xmag = 2 * np.abs(X) / N
plot_mag_spectrum(Xmag[1:1000], N)
Xmag = Xmag[1:1000]

In [ ]:
fractions   = np.linspace(0.0, len(Xmag) *1.0 /N, num=len(Xmag))
frequencies = np.linspace(0.0, (len(Xmag) *1.0 / N)*44100, num=len(Xmag))

# find the index (dft bin) of the peak
peak_bin = np.argmax(Xmag)
print('Index in bins: ', peak_bin)
print('Index in fractions of sampling rate', fractions[peak_bin])
print('Index in Hz: ', frequencies[peak_bin])
peak_freq = frequencies[peak_bin]
peak_midi = int(69 + 12*np.log2(peak_freq/440.))
print(peak_midi)